# EV Vehicle Distribution Visualization

In [1]:
import pandas as pd

df_LAD_to_ITL = pd.read_csv('./LAD_to_ITL(2021).csv')

In [2]:
df_LAD_to_ITL['NUTS_ID'] = df_LAD_to_ITL['ITL321CD'].astype(str).str.replace('^TL', 'UK', regex=True)
df_LAD_to_ITL.rename(columns={'LAD21CD': 'ONScode'}, inplace=True)
df_LAD_to_ITL.to_csv('./LAD_to_NUTS3.csv', index=False)


In [4]:
# ONS_NUTS3_mapping = dict(zip(df_LAD_to_ITL['ONScode'], df_LAD_to_ITL['NUTS_ID']))

ONS_NUTS3_mapping = df_LAD_to_ITL.groupby('ONScode')['NUTS_ID'].apply(list).to_dict()
ONS_to_NUTS3 = {k: list(dict.fromkeys(v)) for k, v in ONS_NUTS3_mapping.items()}


Car = pd.read_csv('./LicensedPlugInVehiclesCarsVEH0142_UK_DfT_Poly_2023_7344901567442451546.csv')
HGV = pd.read_csv('./LicensedPlugInVehiclesHGVVEH0142_UK_DfT_Poly_2023_-4512007299528620992.csv')
LGV = pd.read_csv('./LicensedPlugInVehiclesLGVVEH0142_UK_DfT_Poly_2023_2502161826204229057.csv')
Buscoach = pd.read_csv('./LicensedPlugInVehiclesBusCoachVEH0142_UK_DfT_Poly_2023_4835146115508310650.csv')

# Car['NUTS_ID'] = Car['ONScode'].map(ONS_NUTS3_mapping)
# HGV['NUTS_ID'] = HGV['ONScode'].map(ONS_NUTS3_mapping)
# LGV['NUTS_ID'] = LGV['ONScode'].map(ONS_NUTS3_mapping)
# Buscoach['NUTS_ID'] = Buscoach['ONScode'].map(ONS_NUTS3_mapping)

df_mapping = pd.DataFrame(list(ONS_to_NUTS3.items()), columns=['ONS_code', 'NUTS3_code'])
df_mapping.to_csv('ONS_NUTS3_mapping.csv', index=False)

In [5]:
population_UK = pd.read_excel('./Population_including_UK.xlsx', sheet_name='UK NUTS 2021 and NUTS 2016')
population_by_NUTS3 = population_UK.groupby('NUTS 3 CODE 2021')['POPULATION'].sum().reset_index()
NUTS3_to_population = dict(zip(population_by_NUTS3['NUTS 3 CODE 2021'], population_by_NUTS3['POPULATION']))

In [6]:
for _, row in df_mapping.iterrows():
    if len(row['NUTS3_code'])>1:
        print(f"Multiple NUTS3 codes for ONS code {row['ONS_code']}: {row['NUTS3_code']}")

Multiple NUTS3 codes for ONS code S12000017: ['UKM61', 'UKM62', 'UKM63']
Multiple NUTS3 codes for ONS code S12000021: ['UKM63', 'UKM93']
Multiple NUTS3 codes for ONS code S12000035: ['UKM63', 'UKM81']


In [8]:
df_mapping = df_mapping.rename(columns={
    "ONS_code": "ONScode",
    "NUTS3_code": "NUTS_ID"
})

df_population = population_by_NUTS3.rename(columns={
    "NUTS 3 CODE 2021": "NUTS_ID",
    "POPULATION": "Population"
})

# Car
df_merged_car = Car.merge(df_mapping, on="ONScode", how="left")
df_exploded_car = df_merged_car.explode("NUTS_ID")
df_exploded_car = df_exploded_car.merge(df_population, on="NUTS_ID", how="left")
df_exploded_car["LAD_total_pop"] = df_exploded_car.groupby("ONScode")["Population"].transform("sum")
df_exploded_car["pop_weight"] = df_exploded_car["Population"] / df_exploded_car["LAD_total_pop"]
df_exploded_car["EV_distributed"] = df_exploded_car["Total (Total)"] * df_exploded_car["pop_weight"]
nuts3_total_car = df_exploded_car.groupby("NUTS_ID")["EV_distributed"].sum().reset_index()

# HGV
df_merged_hgv = HGV.merge(df_mapping, on="ONScode", how="left")
df_exploded_hgv = df_merged_hgv.explode("NUTS_ID")
df_exploded_hgv = df_exploded_hgv.merge(df_population, on="NUTS_ID", how="left")
df_exploded_hgv["LAD_total_pop"] = df_exploded_hgv.groupby("ONScode")["Population"].transform("sum")
df_exploded_hgv["pop_weight"] = df_exploded_hgv["Population"] / df_exploded_hgv["LAD_total_pop"]
df_exploded_hgv["EV_distributed"] = df_exploded_hgv["Total (Total)"] * df_exploded_hgv["pop_weight"]
nuts3_total_hgv = df_exploded_hgv.groupby("NUTS_ID")["EV_distributed"].sum().reset_index()

# LGV
df_merged_lgv = LGV.merge(df_mapping, on="ONScode", how="left")
df_exploded_lgv = df_merged_lgv.explode("NUTS_ID")
df_exploded_lgv = df_exploded_lgv.merge(df_population, on="NUTS_ID", how="left")
df_exploded_lgv["LAD_total_pop"] = df_exploded_lgv.groupby("ONScode")["Population"].transform("sum")
df_exploded_lgv["pop_weight"] = df_exploded_lgv["Population"] / df_exploded_lgv["LAD_total_pop"]
df_exploded_lgv["EV_distributed"] = df_exploded_lgv["Total (Total)"] * df_exploded_lgv["pop_weight"]
nuts3_total_lgv = df_exploded_lgv.groupby("NUTS_ID")["EV_distributed"].sum().reset_index()

# Buscoach
df_merged_buscoach = Buscoach.merge(df_mapping, on="ONScode", how="left")
df_exploded_buscoach = df_merged_buscoach.explode("NUTS_ID")
df_exploded_buscoach = df_exploded_buscoach.merge(df_population, on="NUTS_ID", how="left")
df_exploded_buscoach["LAD_total_pop"] = df_exploded_buscoach.groupby("ONScode")["Population"].transform("sum")
df_exploded_buscoach["pop_weight"] = df_exploded_buscoach["Population"] / df_exploded_buscoach["LAD_total_pop"]
df_exploded_buscoach["EV_distributed"] = df_exploded_buscoach["Total (Total)"] * df_exploded_buscoach["pop_weight"]
nuts3_total_buscoach = df_exploded_buscoach.groupby("NUTS_ID")["EV_distributed"].sum().reset_index()


In [ ]:
# Car_numeric_cols = Car.select_dtypes(include='number').columns
# Car_NUTS3 = Car.groupby('NUTS_ID').agg({
#     'ONSname': 'first',
#     'Region': 'first',
#     **{col: 'sum' for col in Car_numeric_cols}
# }).reset_index()

# HGV_numeric_cols = HGV.select_dtypes(include='number').columns
# HGV_NUTS3 = HGV.groupby('NUTS_ID').agg({
#     'ONSname': 'first',
#     'Region': 'first',
#     **{col: 'sum' for col in HGV_numeric_cols}
# }).reset_index()

# LGV_numeric_cols = LGV.select_dtypes(include='number').columns
# LGV_NUTS3 = LGV.groupby('NUTS_ID').agg({
#     'ONSname': 'first',
#     'Region': 'first',
#     **{col: 'sum' for col in LGV_numeric_cols}
# }).reset_index()

# Buscoach_numeric_cols = Buscoach.select_dtypes(include='number').columns
# Buscoach_NUTS3 = Buscoach.groupby('NUTS_ID').agg({
#     'ONSname': 'first',
#     'Region': 'first',
#     **{col: 'sum' for col in Buscoach_numeric_cols}
# }).reset_index()


In [ ]:
# Car_NUTS3['weight'] = Car_NUTS3['Total (Total)']/ Car_NUTS3['Total (Total)'].sum()
# HGV_NUTS3['weight'] = HGV_NUTS3['Total (Total)']/ HGV_NUTS3['Total (Total)'].sum()
# LGV_NUTS3['weight'] = LGV_NUTS3['Total (Total)']/ LGV_NUTS3['Total (Total)'].sum()
# Buscoach_NUTS3['weight'] = Buscoach_NUTS3['Total (Total)']/ Buscoach_NUTS3['Total (Total)'].sum()
# Car_NUTS3.to_csv('Car_NUTS3.csv', index=False)


In [ ]:
import folium
from folium.plugins import HeatMap
import geopandas as gpd

gdf_nuts = gpd.read_file("ref-nuts-2021-01m.geojson/NUTS_RG_01M_2021_4326_LEVL_3.geojson")
gdf_nuts = gdf_nuts[gdf_nuts['CNTR_CODE'] == 'UK']

gdf_merged_car = gdf_nuts.merge(nuts3_total_car, left_on='NUTS_ID', right_on='NUTS_ID')
m_car = folium.Map(location=[54.0, -2.0], zoom_start=6, tiles="CartoDB positron")

# Heat Map - Car

In [ ]:
gdf_merged_car['lat'] = gdf_merged_car.geometry.centroid.y
gdf_merged_car['lon'] = gdf_merged_car.geometry.centroid.x

heat_data = [[row['lat'], row['lon'], row['weight']] for index, row in gdf_merged.iterrows()]

HeatMap(heat_data, radius=15, blur=10, max_zoom=8).add_to(m)
m_car.save("nuts3_Car_heatmap.html")

/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_44344/1495539863.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_merged['lat'] = gdf_merged.geometry.centroid.y
/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_44344/1495539863.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_merged['lon'] = gdf_merged.geometry.centroid.x


# Choropleth Map - Car

In [ ]:
from branca.colormap import linear
import numpy as np

gdf_merged_car['log_total'] = np.log1p(gdf_merged_car['EV_distributed'])

colormap = linear.OrRd_09.scale(gdf_merged_car['log_total'].min(), gdf_merged_car['log_total'].max())

geojson = folium.GeoJson(
    gdf_merged_car,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['log_total']) if feature['properties']['log_total'] is not None else 'white',
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NUTS_ID", "EV_distributed"],
        aliases=["区域编号", "数量"],
        localize=True
    ),
    name="区域数量图"
).add_to(m_car)

# 添加色带图例
colormap.caption = '数量'
colormap.add_to(m_car)

m_car.save('nuts3_Car_choropleth.html')

In [11]:
gdf_merged_hgv = gdf_nuts.merge(nuts3_total_hgv, left_on='NUTS_ID', right_on='NUTS_ID')
m_hgv = folium.Map(location=[54.0, -2.0], zoom_start=6, tiles="CartoDB positron")

gdf_merged_hgv['log_total'] = np.log1p(gdf_merged_hgv['EV_distributed'])

colormap = linear.OrRd_09.scale(gdf_merged_hgv['log_total'].min(), gdf_merged_hgv['log_total'].max())

geojson = folium.GeoJson(
    gdf_merged_hgv,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['log_total']) if feature['properties']['log_total'] is not None else 'white',
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NUTS_ID", "EV_distributed"],
        aliases=["区域编号", "数量"],
        localize=True
    ),
    name="区域数量图"
).add_to(m_hgv)

# 添加色带图例
colormap.caption = '数量'
colormap.add_to(m_hgv)

m_hgv.save('nuts3_HGV_choropleth.html')


In [12]:
gdf_merged_lgv = gdf_nuts.merge(nuts3_total_lgv, left_on='NUTS_ID', right_on='NUTS_ID')
m_lgv = folium.Map(location=[54.0, -2.0], zoom_start=6, tiles="CartoDB positron")

gdf_merged_lgv['log_total'] = np.log1p(gdf_merged_lgv['EV_distributed'])

colormap = linear.OrRd_09.scale(gdf_merged_lgv['log_total'].min(), gdf_merged_lgv['log_total'].max())

geojson = folium.GeoJson(
    gdf_merged_lgv,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['log_total']) if feature['properties']['log_total'] is not None else 'white',
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NUTS_ID", "EV_distributed"],
        aliases=["区域编号", "数量"],
        localize=True
    ),
    name="区域数量图"
).add_to(m_lgv)

# 添加色带图例
colormap.caption = '数量'
colormap.add_to(m_lgv)

m_lgv.save('nuts3_LGV_choropleth.html')

In [13]:
gdf_merged_buscoach = gdf_nuts.merge(nuts3_total_buscoach, left_on='NUTS_ID', right_on='NUTS_ID')
m_buscoach = folium.Map(location=[54.0, -2.0], zoom_start=6, tiles="CartoDB positron")

gdf_merged_buscoach['log_total'] = np.log1p(gdf_merged_buscoach['EV_distributed'])

colormap = linear.OrRd_09.scale(gdf_merged_buscoach['log_total'].min(), gdf_merged_buscoach['log_total'].max())

geojson = folium.GeoJson(
    gdf_merged_buscoach,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['log_total']) if feature['properties']['log_total'] is not None else 'white',
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NUTS_ID", "EV_distributed"],
        aliases=["区域编号", "数量"],
        localize=True
    ),
    name="区域数量图"
).add_to(m_buscoach)

# 添加色带图例
colormap.caption = '数量'
colormap.add_to(m_buscoach)

m_buscoach.save('nuts3_Buscoach_choropleth.html')